native rag with hybird search (vector + bm25)

In [11]:
import os
import re
from typing import Any, Dict

import mlflow
from rich import print as rptint
from dotenv import load_dotenv
import numpy as np
from pymongo import MongoClient

from langchain.agents import create_agent
from langchain_core.documents import Document
from langchain.agents.middleware import AgentMiddleware, AgentState
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_mongodb import MongoDBAtlasVectorSearch
from langchain_mongodb.retrievers.hybrid_search import MongoDBAtlasHybridSearchRetriever

mlflow.langchain.autolog()
load_dotenv()

MONGODB_URI = os.getenv('MONGODB_URI')
DB_NAME = 'agent_db'
COLLECTION_NAME = 'cxi-travel-ins-collection'
ATLAS_VECTOR_SEARCH_INDEX_NAME = "cxi-travel-ins-qa-vector-index"

client = MongoClient(MONGODB_URI)
MONGODB_COLLECTION = client[DB_NAME][COLLECTION_NAME]

In [2]:
client = MongoClient(MONGODB_URI)
embeddings = GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-001')
llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash'
)
vector_store = MongoDBAtlasVectorSearch(
    collection=MONGODB_COLLECTION,
    embedding=embeddings,
    index_name=ATLAS_VECTOR_SEARCH_INDEX_NAME,
    relevance_score_fn="cosine",
    embedding_key="embedding",
    text_key="text"
)
retriever = MongoDBAtlasHybridSearchRetriever(
    vectorstore=vector_store,
    search_index_name="search_index",
    top_k=5,
    fulltext_penalty=50,
    vector_penalty=50,
    post_filter=[
        {
            '$project': {
                'embedding': 0,
            }
        }
    ]
)

In [ ]:
class State(AgentState):
    context: list[Document]
    
class RetrieveDocumentsMiddleware(AgentMiddleware[State]):
    state_schema = State
    
    def before_model(self, state: AgentState) -> dict[str, Any] | None:
        last_message = state['messages'][-1]
        retrieved_docs = vector_store.similarity_search(last_message.content, k=5)
        
        docs = "\n\n".join([doc.page_content for doc in retrieved_docs])
        
        augmented_context = (
            f"{last_message.content}\n\n"
            f"retrevel result: {docs}\n\n"
            "Use the above result to answer the user's question."
            "If you don't know the answer, just say 'I don't know'."
            "使用繁體中文回答"
        )
        
        return {
            "messages": [last_message.model_copy(update={"content": augmented_context})],
            "context": retrieved_docs,
        }

In [ ]:
rag_llm = create_agent(
    model=llm,
    tools=[],
    middleware=[RetrieveDocumentsMiddleware()]
)

In [29]:
rag_llm.invoke({"messages": [{"role": "user", "content": "行李遺失後應該如何申請理賠？"}]})

{'messages': [HumanMessage(content="行李遺失後應該如何申請理賠？\n\nretrevel result: 第四十三條 行李損失保險理賠文件 \n被保險人向本公司申請理賠時，應檢具下列文件： \n一、理賠申請書。 \n二、因第三十九條第一項第一款所列事故申請理賠者：向警方報案證明。 \n三、因第三十九條第一項第二款所列事故申請理賠者：公共交通工具業者所開立之事故與損失\n證明。\n\n第三十八條 行李延誤保險理賠文件 \n被保險人向本公司申請理賠時，應檢具下列文件： \n一、理賠申請書。 \n二、公共交通工具業者所出具行李延誤達六小時以上之文件。\n\n第四十四條 追回處理 \n本公司因行李遭竊盜、強盜、搶奪或遺失事故為理賠後，其所有權歸本公司，如經追回，被保\n險人願意收回時，被保險人應將該項保險標的物之賠償金額返還本公司。\n\n第五十七條 信用卡盜用保險承保範圍 \n被保險人於海外旅行期間內，因其所持有之信用卡遺失或遭受竊盜、搶奪而向該信用卡之發行\n機構掛失或止付前三十六個小時內，因未經授權而遭盜刷之損失，包括信用卡掛失止付及申請\n重置之費用，本公司依本保險契約之約定對被保險人負理賠之責。 \n前項之損失及費用應扣除該信用卡之發行機構就該信用卡之遺失或遭受竊盜、搶奪事件依約應\n承擔之部分。 \n被保險人應於知悉信用卡遭受竊盜、搶奪後立即向當地警察機關報案並取得報案證明。但自行\n遺失者不在此限。\n\n第四十二條 行李損失保險事故發生時之處理 \n發生本承保範圍第三十九條第一項第一款所列事故時，被保險人應在二十四小時內，向當地警\n政單位報案並取得報案證明。 \n發生本承保範圍第三十九條第一項第二款所列事故時，被保險人應儘速通知公共交通工具業者 \n，並向其索取書面事故與損失證明。\n\nUse the above result to answer the user's question.If you don't know the answer, just say 'I don't know'.使用繁體中文回答", additional_kwargs={}, response_metadata={}, id='49257f03-5d82-4232-a2ba-1de1cb48af89'),
  AIMessage(cont